### Imports

In [ ]:
from helpers import load_csv_dataset, save_csv_dataset
from pathlib import Path
import pandas as pd
from scipy import stats

data_dir = Path('../data')
quotations_dataset = load_csv_dataset(data_dir.joinpath('quotations.csv'), delimiter = ';')
respondents_dataset = load_csv_dataset(data_dir.joinpath('respondents.csv'), delimiter = ';')
requirements_dataset = load_csv_dataset(data_dir.joinpath('requirements.csv'))

requirements_and_respondents_dataset = []
for r in requirements_dataset:
    respondents_per_code = set([x['document'] for x in quotations_dataset if r['code'] in x['codes']])
    for respondent in respondents_per_code:
        id = [r1['role'] for r1 in respondents_dataset if r1['id'] == respondent][0]
        requirements_and_respondents_dataset.append({'scenario': r['scenario'], 
                        'category': r['category'], 
                        'complete-requirement': r['complete-requirement'], 
                        'code': r['code'],
                        'respondent-id':respondent,
                        'respondent-role': [r1['role'] for r1 in respondents_dataset if r1['id'] == respondent][0],
                        'respondent-experience':[r1['experience'] for r1 in respondents_dataset if r1['id'] == respondent][0],
                        'respondent-education':[r1['education'] for r1 in respondents_dataset if r1['id'] == respondent][0]})

save_csv_dataset("../data/correlation-respondents-requirements/requirements-and-respondents.csv", requirements_and_respondents_dataset)
requirements_dataset = pd.read_csv("../data/correlation-respondents-requirements/requirements-and-respondents.csv")

### Chi-square Test

#### 1. Test Category vs Charachtericts

In [ ]:
#Test Role
crosstab = pd.crosstab(requirements_dataset['category'], requirements_dataset['respondent-role'])
stats.chi2_contingency(crosstab)

In [ ]:
#Test Experience
crosstab = pd.crosstab(requirements_dataset['category'], requirements_dataset['respondent-experience'])
stats.chi2_contingency(crosstab)

In [ ]:
#Test Experience
crosstab = pd.crosstab(requirements_dataset['category'], requirements_dataset['respondent-education'])
stats.chi2_contingency(crosstab)

#### 2. Test Requirements vs Charachtericts

In [ ]:
#Test Role
crosstab = pd.crosstab(requirements_dataset['complete-requirement'], requirements_dataset['respondent-role'])
stats.chi2_contingency(crosstab)

In [ ]:
#Test Experience
crosstab = pd.crosstab(requirements_dataset['complete-requirement'], requirements_dataset['respondent-experience'])
stats.chi2_contingency(crosstab)

In [ ]:
#Test Education
crosstab = pd.crosstab(requirements_dataset['complete-requirement'], requirements_dataset['respondent-education'])
stats.chi2_contingency(crosstab)

### Relating requirements categories and respondents

In [ ]:
dataset = []
quotations_dataset = load_csv_dataset(data_dir.joinpath('quotations.csv'), delimiter = ';')
respondents_dataset = load_csv_dataset(data_dir.joinpath('respondents.csv'), delimiter = ';')
requirements_dataset = load_csv_dataset(data_dir.joinpath('requirements.csv'))
requirements_categories = list(set([x['category'] for x in requirements_dataset]))

for respondent in respondents_dataset:
    categories = []
    for r in requirements_dataset:
        respondents_per_code = set([x['document'] for x in quotations_dataset if r['code'] in x['codes']])
        if respondent['id'] in respondents_per_code:
            categories.append(r['category'])
    row = {'respondent-id': respondent['id'],
           'respondent-role': respondent['role'],
           'respondent-experience': respondent['experience'],
           'respondent-education': respondent['education']}
    
    for cat in requirements_categories:
        if cat in categories:
            row[cat] = 1
        else:
            row[cat] = 0
            
    dataset.append(row)
save_csv_dataset('../data/correlation-respondents-requirements/respondents-per-requirement-category.csv', dataset)

### 3d Scatter Plots

In [ ]:
!pip install plotly
import pandas as pd
import plotly.express as px
df = pd.read_csv("../data/correlation-respondents-requirements/respondents-per-requirement-category.csv")

categories = df.keys()[-7:]

for i in categories:
    d = df.loc[df[i] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id']].groupby(['respondent-experience','respondent-role','respondent-education'], as_index=False).count().rename(columns={'respondent-id': 'n-respondents'})
    
    fig = px.scatter_3d(d, x='respondent-role', y='respondent-experience', z='respondent-education',
                        labels={
                         "respondent-role": "Role",
                         "respondent-experience": "Experience",
                         "respondent-education": "Education Level"
                     }, size = d['n-respondents'], title=f"Number of respondents, per profile - {i}")
    
    fig.write_html(f"../data/correlation-respondents-requirements/scatter-plots/respondents-{i.replace('/','-').replace(' ','-')}.html")

In [ ]:
!pip install plotly
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
df = pd.read_csv("../data/respondents-per-requirement-category.csv")

categories = df.keys()[-7:]

roles = ['Phd Student', 'Student', 'Researcher', 'Software Developer', 'Customer Engineer', 'Quality Assurance', 'Software Architect/Designer','Project/Engineering Manager']
experiences = ['less than 1 year','1 - 5 years','6 - 10 years','more than 10 years']
education = ['Unfinished bachelor', 'Bachelor','Master', 'PhD']

for i in categories:
    d = df.loc[df[i] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id']].groupby(['respondent-experience','respondent-role','respondent-education'], as_index=False).count().rename(columns={'respondent-id': 'n-respondents'})
    fig = go.Figure(data=go.Scatter3d(
    x=d['respondent-role'],
    y=d['respondent-experience'],
    z=d['respondent-education'],
    #text=df['country'][start:end],
    mode='markers',
    marker=dict(
        sizemode='diameter',
        sizeref=1,
        size=d['n-respondents'] * 10
        )
    ))

    fig.update_layout(scene = dict(
                    xaxis = dict(
                         backgroundcolor="rgb(200, 200, 230)",
                         gridcolor="white",
                         showbackground=True,
                         zerolinecolor="white",),
                    yaxis = dict(
                        backgroundcolor="rgb(230, 200,230)",
                        gridcolor="white",
                        showbackground=True,
                        zerolinecolor="white"),
                    zaxis = dict(
                        backgroundcolor="rgb(230, 230,200)",
                        gridcolor="white",
                        showbackground=True,
                        zerolinecolor="white",),),
                    width=400,
                    margin=dict(
                    r=10, l=10,
                    b=10, t=10)
                  )

    fig.update_xaxes(categoryorder = 'array', categoryarray=roles)
    fig.update_yaxes(categoryorder = 'array', categoryarray=experiences)
    #fig.update_zaxes(categoryorder = 'array', categoryarray=education)
    
    fig.write_html(f"../data/correlation-respondents-requirements/scatter-plots/respondents-{i.replace('/','-').replace(' ','-')}.html")

### Scatter plots with symbols

In [ ]:
!pip install plotly
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

def combine_plotly_figs_to_html(plotly_figs, html_fname, include_plotlyjs='cdn', 
                                separator=None, auto_open=False):
    with open(html_fname, 'w') as f:
        f.write(plotly_figs[0].to_html(include_plotlyjs=include_plotlyjs))
        for fig in plotly_figs[1:]:
            if separator:
                f.write(separator)
            f.write(fig.to_html(full_html=False, include_plotlyjs=False))

    if auto_open:
        import pathlib, webbrowser
        uri = pathlib.Path(html_fname).absolute().as_uri()
        webbrowser.open(uri)


df = pd.read_csv("../data/respondents-per-requirement-category.csv")
categories = ['information to be provided/detailed information', 
              'information to be provided/grouped information',
              'tool usage/tool execution/workflow execution',
              'tool usage/tool execution/execution criteria',
              'tool usage/tool execution/manual execution',
              'tool usage/tool interface',
              'tool usage/customization']

roles = ['Phd Student', 'Student', 'Researcher', 'Software Developer', 'Customer Engineer', 'Quality Assurance', 'Software Architect/Designer','Project/Engineering Manager']
experiences = ['less than 1 year','1 - 5 years','6 - 10 years','more than 10 years']
education = ['Unfinished bachelor', 'Bachelor','Master', 'PhD']
figures = []

for i in categories:
    d = df.loc[df[i] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id']].groupby(['respondent-experience','respondent-role','respondent-education'], as_index=False).count().rename(columns={'respondent-id': 'n-respondents'})
    fig = px.scatter(d, x='respondent-role', 
                     y='respondent-experience',
                     symbol='respondent-education',
                     size='n-respondents',
                     color='respondent-education',
                     width=800, 
                     height=400,
                    opacity=0.7, 
                    title=i)
    fig.update_traces(opacity=0.7)

    figures.append(fig)
    #fig.write_html(f"../data/scatter-plots/respondents-{i.replace('/','-').replace(' ','-')}.html")

combine_plotly_figs_to_html(figures, '../data/correlation-respondents-requirements/scatter-plots/plots.html')

### information to be provided/detailed information

In [ ]:
p = df.loc[df['information to be provided/detailed information'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

### information to be provided/grouped information

In [ ]:
p = df.loc[df['information to be provided/grouped information'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

### tool usage/tool execution/workflow execution

In [ ]:
p = df.loc[df['tool usage/tool execution/workflow execution'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

### tool usage/tool execution/threshold execution

In [ ]:
p = df.loc[df['tool usage/tool execution/execution criteria'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

### tool usage/tool execution/manual execution

In [ ]:
p = df.loc[df['tool usage/tool execution/manual execution'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

### tool usage/tool interface

In [ ]:
p = df.loc[df['tool usage/tool interface'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()

### tool usage/customization

In [ ]:
p = df.loc[df['tool usage/customization'] == 1][['respondent-experience','respondent-role','respondent-education','respondent-id','information to be provided/detailed information']]
p.groupby(['respondent-experience','respondent-role','respondent-education','respondent-id']).count()